# Task 2: Predictive Linear Regression Pipeline
**Progree - Machine Learning Internship**

**Objective:** Build a supervised regression model that predicts a continuous value.

**Dataset:** California Housing (scikit-learn). Target = median house value in units of $100,000.

**Pipeline:** load data -> explore -> preprocessing (mean imputation, feature scaling, categorical encoding) -> Linear Regression baseline -> Ridge regression (tuned alpha) -> log R2 and RMSE -> plots.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error

SEED = 42
np.random.seed(SEED)

## 2. Load the dataset
The data is downloaded by scikit-learn the first time (needs internet, which Colab has).

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
TARGET = 'MedHouseVal'

print('Shape:', df.shape)
display(df.head())
display(df.describe().T)
print('Missing values in raw data:', int(df.isna().sum().sum()))

## 3. Feature engineering for the categorical step
California Housing has only numeric columns and no missing values. To demonstrate every preprocessing step the task asks for, I add one categorical feature, `HouseAgeGroup` (new / mid / old), made from `HouseAge`. The pipeline still contains a mean imputer, so it would handle missing values if the data had any.

In [ ]:
df['HouseAgeGroup'] = pd.cut(df['HouseAge'], bins=[0, 15, 30, 100],
                             labels=['new', 'mid', 'old']).astype(str)

X = df.drop(columns=[TARGET])
y = df[TARGET]

numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = ['HouseAgeGroup']
print('Numeric features    :', numeric_cols)
print('Categorical features:', categorical_cols)
print(X['HouseAgeGroup'].value_counts())

## 4. Exploration

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(y, bins=50, color='#2a7de1')
ax[0].set_title('Target distribution (x $100,000)'); ax[0].set_xlabel('MedHouseVal')
corr = df[numeric_cols + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
ax[1].barh(corr.index, corr.values, color='#2a7de1')
ax[1].set_title('Correlation with target')
plt.tight_layout(); plt.savefig('t2_exploration.png', dpi=150); plt.show()

## 5. Train / test split and preprocessing pipeline
The split happens first, so the imputer and scaler learn only from training data (no leakage).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print('Train:', X_train.shape, ' Test:', X_test.shape)

numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),   # mean imputation
    ('scale', StandardScaler()),                  # feature scaling
])
categorical_pipe = Pipeline([
    ('encode', OneHotEncoder(handle_unknown='ignore')),   # categorical encoding
])
preprocess = ColumnTransformer([
    ('num', numeric_pipe, numeric_cols),
    ('cat', categorical_pipe, categorical_cols),
])

## 6. Baseline: Linear Regression

In [ ]:
lin = Pipeline([('prep', preprocess), ('model', LinearRegression())])
lin.fit(X_train, y_train)

## 7. Regularised model: Ridge Regression with tuned alpha
GridSearchCV tries many regularisation strengths using 5-fold cross-validation on the training set.

In [ ]:
ridge = Pipeline([('prep', preprocess), ('model', Ridge(random_state=SEED))])
param_grid = {'model__alpha': np.logspace(-3, 3, 13)}
search = GridSearchCV(ridge, param_grid, cv=KFold(5, shuffle=True, random_state=SEED),
                      scoring='neg_root_mean_squared_error')
search.fit(X_train, y_train)
best_ridge = search.best_estimator_
print('Best alpha:', search.best_params_['model__alpha'])
print('Best CV RMSE:', round(-search.best_score_, 4))

## 8. Evaluation: R2 and RMSE

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

rows = []
for name, mdl in [('Linear Regression', lin), ('Ridge Regression (tuned)', best_ridge)]:
    for split, Xs, ys in [('train', X_train, y_train), ('test', X_test, y_test)]:
        pred = mdl.predict(Xs)
        rows.append({'Model': name, 'Split': split,
                     'R2': round(r2_score(ys, pred), 4),
                     'RMSE': round(rmse(ys, pred), 4)})
metrics = pd.DataFrame(rows)
display(metrics)

cv_r2 = cross_val_score(best_ridge, X_train, y_train,
                        cv=KFold(5, shuffle=True, random_state=SEED), scoring='r2')
print('Ridge 5-fold CV R2 on training data:', np.round(cv_r2, 4), '| mean =', round(cv_r2.mean(), 4))
print('RMSE is in units of $100,000, so an RMSE of 0.7 means a typical error of about $70,000.')
metrics.to_csv('t2_metrics.csv', index=False)

## 9. Plots

In [ ]:
pred = best_ridge.predict(X_test)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

ax[0].scatter(y_test, pred, s=4, alpha=0.3, color='#2a7de1')
lims = [0, 5.2]; ax[0].plot(lims, lims, 'r--', label='perfect prediction')
ax[0].set_xlabel('Actual'); ax[0].set_ylabel('Predicted'); ax[0].set_title('Actual vs predicted (test)'); ax[0].legend()

ax[1].hist(y_test - pred, bins=60, color='#2a7de1')
ax[1].set_title('Residuals (actual - predicted)'); ax[1].set_xlabel('Error')

names = best_ridge.named_steps['prep'].get_feature_names_out()
coefs = pd.Series(best_ridge.named_steps['model'].coef_, index=names).sort_values()
ax[2].barh([n.split('__')[-1] for n in coefs.index], coefs.values, color='#2a7de1')
ax[2].set_title('Ridge coefficients (scaled features)')
plt.tight_layout(); plt.savefig('t2_results.png', dpi=150); plt.show()

## 10. Conclusion
- The pipeline performs mean imputation, standard scaling and one-hot encoding inside one `Pipeline`, so the exact same steps are applied to training and new data.
- Ridge regression adds an L2 penalty; the alpha was chosen by cross-validation. Because California Housing has only a few features, Ridge and plain Linear Regression usually end up with very similar scores. That is a normal result, not a bug.
- A purely linear model cannot capture non-linear effects such as location, which limits R2.
- **Future work:** add Lasso/ElasticNet, polynomial features, or tree-based models (Random Forest, Gradient Boosting) and compare their R2 and RMSE.